In [80]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score

## Affin

In [68]:
def load_afinn(path):
    afinn = {}
    with open(path, "r") as f:
        for line in f:
            word, score = line.strip().split("\t")
            afinn[word] = int(score)
    return afinn

In [69]:
NEGATIONS = {"not", "no", "never", "n't"}

def afinn_predict(text, afinn_dict):
    words = text.lower().split()
    score = 0
    negate = False

    for word in words:
        if word in NEGATIONS:
            negate = True
            continue

        word_score = afinn_dict.get(word, 0)

        if negate:
            word_score *= -1
            negate = False

        score += word_score

    # Convert score → class
    if score > 0:
        return 1
    elif score < 0:
        return -1
    else:
        return 0



In [70]:
schemas = ["fully_cleaned", "without_lemmatization", "without_stopwords_lowercasing_punct"]

for schema in schemas:
    df = pd.read_csv("schemas/"+schema+".csv")
    # Load dictionary
    afinn = load_afinn("AFINN-111.txt")
    df[f"afinn_pred_{schema}"] = df["cleaned_review"].apply(lambda x: afinn_predict(str(x), afinn))
    df[f"afinn_pred_{schema}"] = df[f"afinn_pred_{schema}"].map({-1: "negative", 0: "neutral", 1: "positive"})
    df.to_csv(f"output/afinn_predictions_{schema}.csv", index=False)

In [81]:
df = pd.read_csv('output/afinn_predictions_fully_cleaned.csv')
print(classification_report(df['sentiment'], df['afinn_pred_fully_cleaned']))
print(confusion_matrix(df['sentiment'], df['afinn_pred_fully_cleaned']))
print("AFINN Fully Cleaned F1 Score:", f1_score(df['sentiment'], df['afinn_pred_fully_cleaned'], average='weighted'))
print("AFINN Fully Cleaned Precision:", precision_score(df['sentiment'], df['afinn_pred_fully_cleaned'], average='weighted'))
print("AFINN Fully Cleaned Recall:", recall_score(df['sentiment'], df['afinn_pred_fully_cleaned'], average='weighted'))
print("AFINN Fully Cleaned Accuracy Score:", accuracy_score(df['sentiment'], df['afinn_pred_fully_cleaned']))


              precision    recall  f1-score   support

    negative       0.43      0.50      0.46         6
     neutral       0.00      0.00      0.00        11
    positive       0.74      0.94      0.83        33

    accuracy                           0.68        50
   macro avg       0.39      0.48      0.43        50
weighted avg       0.54      0.68      0.60        50

[[ 3  0  3]
 [ 3  0  8]
 [ 1  1 31]]
AFINN Fully Cleaned F1 Score: 0.6009846153846155
AFINN Fully Cleaned Precision: 0.5385714285714286
AFINN Fully Cleaned Recall: 0.68
AFINN Fully Cleaned Accuracy Score: 0.68


In [82]:
df = pd.read_csv('output/afinn_predictions_without_lemmatization.csv')
print(classification_report(df['sentiment'], df['afinn_pred_without_lemmatization']))
print(confusion_matrix(df['sentiment'], df['afinn_pred_without_lemmatization']))
print("AFINN Without Lemmatization F1 Score:", f1_score(df['sentiment'], df['afinn_pred_without_lemmatization'], average='weighted'))
print("AFINN Without Lemmatization Precision:", precision_score(df['sentiment'], df['afinn_pred_without_lemmatization'], average='weighted'))
print("AFINN Without Lemmatization Recall:", recall_score(df['sentiment'], df['afinn_pred_without_lemmatization'], average='weighted'))

              precision    recall  f1-score   support

    negative       0.50      0.50      0.50         6
     neutral       0.00      0.00      0.00        11
    positive       0.72      0.94      0.82        33

    accuracy                           0.68        50
   macro avg       0.41      0.48      0.44        50
weighted avg       0.54      0.68      0.60        50

[[ 3  0  3]
 [ 2  0  9]
 [ 1  1 31]]
AFINN Without Lemmatization F1 Score: 0.5984210526315789
AFINN Without Lemmatization Precision: 0.535813953488372
AFINN Without Lemmatization Recall: 0.68


In [85]:
df = pd.read_csv('output/afinn_predictions_without_stopwords_lowercasing_punct.csv')
print(classification_report(df['sentiment'], df['afinn_pred_without_stopwords_lowercasing_punct']))
print(confusion_matrix(df['sentiment'], df['afinn_pred_without_stopwords_lowercasing_punct']))
print("AFINN Without Stopwords, Lowercasing, Punctuation F1 Score:", f1_score(df['sentiment'], df['afinn_pred_without_stopwords_lowercasing_punct'], average='weighted'))
print("AFINN Without Stopwords, Lowercasing, Punctuation Precision:", precision_score(df['sentiment'], df['afinn_pred_without_stopwords_lowercasing_punct'], average='weighted'))
print("AFINN Without Stopwords, Lowercasing, Punctuation Recall:", recall_score(df['sentiment'], df['afinn_pred_without_stopwords_lowercasing_punct'], average='weighted'))
print("AFINN Without Stopwords, Lowercasing, Punctuation Accuracy Score:", accuracy_score(df['sentiment'], df['afinn_pred_without_stopwords_lowercasing_punct']))

              precision    recall  f1-score   support

    negative       0.75      0.50      0.60         6
     neutral       0.00      0.00      0.00        11
    positive       0.71      0.97      0.82        33

    accuracy                           0.70        50
   macro avg       0.49      0.49      0.47        50
weighted avg       0.56      0.70      0.61        50

[[ 3  0  3]
 [ 1  0 10]
 [ 0  1 32]]
AFINN Without Stopwords, Lowercasing, Punctuation F1 Score: 0.6135384615384615
AFINN Without Stopwords, Lowercasing, Punctuation Precision: 0.5593333333333333
AFINN Without Stopwords, Lowercasing, Punctuation Recall: 0.7
AFINN Without Stopwords, Lowercasing, Punctuation Accuracy Score: 0.7


## Vader

In [74]:
import nltk

In [75]:
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def vader_predict(text):
    score = sia.polarity_scores(str(text))["compound"]

    if score > 0.05:
        return 1
    elif score < -0.05:
        return -1
    else:
        return 0

In [76]:
# df["vader_pred"] = df["cleaned_review"].apply(vader_predict)
schemas = ["fully_cleaned", "without_lemmatization", "without_stopwords_lowercasing_punct"]
for schema in schemas:
    df = pd.read_csv("schemas/"+schema+".csv")
    # Load dictionary
    afinn = load_afinn("AFINN-111.txt")
    
    df[f"vader_pred_{schema}"] = df["cleaned_review"].apply(vader_predict)
    df[f"vader_pred_{schema}"] = df[f"vader_pred_{schema}"].map({-1: "negative", 0: "neutral", 1: "positive"})
    df.to_csv(f"output/vader_predictions_{schema}.csv", index=False)

In [86]:
df = pd.read_csv('output/vader_predictions_fully_cleaned.csv')
print(classification_report(df["sentiment"], df["vader_pred_fully_cleaned"]))
print(confusion_matrix(df["sentiment"], df["vader_pred_fully_cleaned"]))
print("VADER Fully Cleaned F1 Score:", f1_score(df['sentiment'], df['vader_pred_fully_cleaned'], average='weighted'))
print("VADER Fully Cleaned Precision:", precision_score(df['sentiment'], df['vader_pred_fully_cleaned'], average='weighted'))
print("VADER Fully Cleaned Recall:", recall_score(df['sentiment'], df['vader_pred_fully_cleaned'], average='weighted'))
print("VADER Fully Cleaned Accuracy Score:", accuracy_score(df['sentiment'], df['vader_pred_fully_cleaned']))

              precision    recall  f1-score   support

    negative       0.75      0.50      0.60         6
     neutral       1.00      0.09      0.17        11
    positive       0.73      1.00      0.85        33

    accuracy                           0.74        50
   macro avg       0.83      0.53      0.54        50
weighted avg       0.79      0.74      0.67        50

[[ 3  0  3]
 [ 1  1  9]
 [ 0  0 33]]
VADER Fully Cleaned F1 Score: 0.6671282051282051
VADER Fully Cleaned Precision: 0.794
VADER Fully Cleaned Recall: 0.74
VADER Fully Cleaned Accuracy Score: 0.74


In [87]:
df = pd.read_csv('output/vader_predictions_without_lemmatization.csv')
print(classification_report(df["sentiment"], df["vader_pred_without_lemmatization"]))
print(confusion_matrix(df["sentiment"], df["vader_pred_without_lemmatization"]))
print("VADER Without Lemmatization F1 Score:", f1_score(df['sentiment'], df['vader_pred_without_lemmatization'], average='weighted'))
print("VADER Without Lemmatization Precision:", precision_score(df['sentiment'], df['vader_pred_without_lemmatization'], average='weighted'))
print("VADER Without Lemmatization Recall:", recall_score(df['sentiment'], df['vader_pred_without_lemmatization'], average='weighted'))
print("VADER Without Lemmatization Accuracy Score:", accuracy_score(df['sentiment'], df['vader_pred_without_lemmatization']))

              precision    recall  f1-score   support

    negative       0.75      0.50      0.60         6
     neutral       0.00      0.00      0.00        11
    positive       0.72      1.00      0.84        33

    accuracy                           0.72        50
   macro avg       0.49      0.50      0.48        50
weighted avg       0.56      0.72      0.62        50

[[ 3  0  3]
 [ 1  0 10]
 [ 0  0 33]]
VADER Without Lemmatization F1 Score: 0.6233924050632912
VADER Without Lemmatization Precision: 0.5634782608695652
VADER Without Lemmatization Recall: 0.72
VADER Without Lemmatization Accuracy Score: 0.72


/home/muhammed_mahmoud/Projects/python_env/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/muhammed_mahmoud/Projects/python_env/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/muhammed_mahmoud/Projects/python_env/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

In [88]:
df = pd.read_csv('output/vader_predictions_without_stopwords_lowercasing_punct.csv')
print(classification_report(df["sentiment"], df["vader_pred_without_stopwords_lowercasing_punct"]))
print(confusion_matrix(df["sentiment"], df["vader_pred_without_stopwords_lowercasing_punct"]))
print("VADER Without Stopwords, Lowercasing, Punctuation F1 Score:", f1_score(df['sentiment'], df['vader_pred_without_stopwords_lowercasing_punct'], average='weighted'))
print("VADER Without Stopwords, Lowercasing, Punctuation Precision:", precision_score(df['sentiment'], df['vader_pred_without_stopwords_lowercasing_punct'], average='weighted'))
print("VADER Without Stopwords, Lowercasing, Punctuation Recall:", recall_score(df['sentiment'], df['vader_pred_without_stopwords_lowercasing_punct'], average='weighted'))
print("VADER Without Stopwords, Lowercasing, Punctuation Accuracy Score:", accuracy_score(df['sentiment'], df['vader_pred_without_stopwords_lowercasing_punct']))

              precision    recall  f1-score   support

    negative       0.43      0.50      0.46         6
     neutral       0.00      0.00      0.00        11
    positive       0.70      0.91      0.79        33

    accuracy                           0.66        50
   macro avg       0.38      0.47      0.42        50
weighted avg       0.51      0.66      0.58        50

[[ 3  0  3]
 [ 1  0 10]
 [ 3  0 30]]
VADER Without Stopwords, Lowercasing, Punctuation F1 Score: 0.5764372469635628
VADER Without Stopwords, Lowercasing, Punctuation Precision: 0.5118936877076412
VADER Without Stopwords, Lowercasing, Punctuation Recall: 0.66
VADER Without Stopwords, Lowercasing, Punctuation Accuracy Score: 0.66


/home/muhammed_mahmoud/Projects/python_env/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/muhammed_mahmoud/Projects/python_env/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/muhammed_mahmoud/Projects/python_env/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag